# 03 — چت‌بات چندنوبته (Multi-Turn Dialogue Manager)

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("."))
from chatbot_common import load_intents, load_schema


## بارگذاری تشخیص‌گر Intent و استخراج‌کننده‌های اسلات

In [ ]:
from chatbot_common import call_llm, extract_json
import difflib

INTENTS = load_intents()
SCHEMA = load_schema()

INTENT_DESCRIPTIONS = {
    "openـaccountـfree": "افتتاح حساب رایگان/قرض‌الحسنه", "openـaccountـcurrent": "افتتاح حساب جاری",
    "openـaccountـdeposit": "افتتاح حساب سپرده/سرمایه‌گذاری", "loanـfree": "درخواست وام قرض‌الحسنه",
    "loanـinterest": "درخواست وام با سود", "card2card": "انتقال وجه کارت به کارت",
    "paya": "انتقال وجه پایا/ساتنا با شبا", "convertـcheque": "نقد کردن یا تبدیل چک",
    "receiptـpayment": "پرداخت قبض یا فاکتور", "installmentـpayment": "پرداخت قسط وام",
    "turnoverـbill": "دریافت گردش حساب", "balanceـbill": "استعلام موجودی حساب",
    "submitـcheque": "ثبت/صدور چک", "receiveـcheque": "دریافت وجه چک",
    "changeـpassword": "تغییر رمز کارت", "duplicateـcard": "درخواست کارت المثنی",
    "closeـcard": "مسدود کردن کارت", "delegateـaccount": "وکالت دادن حساب",
    "currencyـrequest": "درخواست ارز", "softwareـproblem": "مشکل نرم‌افزاری",
    "signinـproblem": "مشکل ورود به حساب",
}


def build_intent_system_prompt():
    lines = [
        "شما یک دستیار طبقه‌بندی قصد برای یک چت‌بات بانکی فارسی هستید.",
        "لیست intent های مجاز:",
    ]
    for intent in INTENTS:
        lines.append(f"- {intent}: {INTENT_DESCRIPTIONS.get(intent, '')}")
    lines.append('فقط این JSON را برگردانید: {"intent": "<یکی از موارد بالا یا unknown>"}')
    return "\n".join(lines)


INTENT_SYSTEM_PROMPT = build_intent_system_prompt()


def classify_intent(text: str) -> str:
    raw = call_llm(INTENT_SYSTEM_PROMPT, text, max_tokens=100, temperature=0.0)
    try:
        intent = extract_json(raw).get("intent", "unknown")
    except Exception:
        intent = "unknown"
    if intent not in INTENTS and intent != "unknown":
        close = difflib.get_close_matches(intent, INTENTS, n=1, cutoff=0.6)
        intent = close[0] if close else "unknown"
    return intent


def extract_all_slots(intent: str, text: str) -> dict:
    slot_defs = SCHEMA.get(intent, [])
    if not slot_defs:
        return {}
    lines = [
        "شما یک استخراج‌کننده‌ی اطلاعات ساختاریافته برای یک چت‌بات بانکی فارسی هستید.",
        f'کاربر قصد "{intent}" را دارد. اسلات‌های مرتبط:',
    ]
    for s in slot_defs:
        lines.append(f"- {s['slot']}: {s['question']}")
    lines.append("فقط اسلات‌هایی را که واقعاً در متن ذکر شده‌اند استخراج کنید؛ حدس نزنید.")
    lines.append('فقط این JSON را برگردانید: {"slots": {"<نام اسلات>": "<مقدار>"}}')
    system_prompt = "\n".join(lines)

    raw = call_llm(system_prompt, text, max_tokens=500, temperature=0.0)
    try:
        return extract_json(raw).get("slots", {})
    except Exception:
        return {}


def extract_single_slot(slot_name: str, question: str, user_reply: str) -> dict:
    system_prompt = (
        "شما یک استخراج‌کننده‌ی مقدار برای یک فیلد در یک فرم بانکی فارسی هستید.\n"
        f"سؤال: «{question}»   نام فیلد: {slot_name}\n"
        "مقدار تمیز و نرمال‌شده را استخراج کنید (بله/خیر -> true/false، اعداد فارسی -> انگلیسی).\n"
        "اگر پاسخ نامعتبر بود، value را null بگذارید.\n"
        'فقط این JSON را برگردانید: {"value": "...", "valid": true}'
    )
    raw = call_llm(system_prompt, user_reply, max_tokens=150, temperature=0.0)
    try:
        parsed = extract_json(raw)
        value = parsed.get("value")
        if value in (None, "null", ""):
            return {"value": None, "valid": False}
        return {"value": value, "valid": bool(parsed.get("valid", True))}
    except Exception:
        return {"value": user_reply.strip(), "valid": True}


## کلاس مدیریت مکالمه

In [ ]:
class MultiTurnChatBot:
    """State machine: awaiting_intent -> asking_slots -> completed"""

    MAX_RETRIES_PER_SLOT = 2

    def __init__(self):
        self.state = "awaiting_intent"
        self.intent = None
        self.slot_defs = []          # لیست {slot, question} برای intent فعلی
        self.filled = {}             # slot_name -> value
        self.pending_index = 0       
        self._retries = {}

    # -------- کمک‌کننده‌ها --------
    def _remaining_slots(self):
        return [s for s in self.slot_defs if s["slot"] not in self.filled]

    def _next_question(self):
        remaining = self._remaining_slots()
        if not remaining:
            return None
        return remaining[0]

    def _merge_extracted(self, extracted: dict):
        valid_slot_names = {s["slot"] for s in self.slot_defs}
        for k, v in extracted.items():
            if k in valid_slot_names and k not in self.filled and v not in (None, ""):
                self.filled[k] = v

    # -------- ورودی اصلی --------
    def process_input(self, user_input: str) -> str:
        if self.state == "awaiting_intent":
            return self._handle_intent_turn(user_input)
        elif self.state == "asking_slots":
            return self._handle_slot_turn(user_input)
        else:  
            return self.generate_json_response()

    def _handle_intent_turn(self, user_input: str) -> str:
        self.intent = classify_intent(user_input)
        if self.intent == "unknown":
            return "متوجه درخواست شما نشدم. می‌تونید دقیق‌تر توضیح بدید چه کاری می‌خواید انجام بدید؟"

        self.slot_defs = SCHEMA.get(self.intent, [])
        self.state = "asking_slots"

        
        if self.slot_defs:
            extracted = extract_all_slots(self.intent, user_input)
            self._merge_extracted(extracted)

        next_q = self._next_question()
        if next_q is None:
            self.state = "completed"
            return f"قصد شناسایی شد: {self.intent}.\n" + self.generate_json_response()

        return f"قصد شناسایی شد: {self.intent}.\n{next_q['question']}"

    def _handle_slot_turn(self, user_input: str) -> str:
        current = self._next_question()
        if current is None:
            self.state = "completed"
            return self.generate_json_response()

        # ۱) تلاش برای گرفتن چند اسلات همزمان از همین پاسخ
        extracted = extract_all_slots(self.intent, user_input)
        self._merge_extracted(extracted)

        # ۲) اگر اسلاتی که الان پرسیده بودیم هنوز خالی است، مستقیماً از همین پاسخ استخراجش کن
        if current["slot"] not in self.filled:
            result = extract_single_slot(current["slot"], current["question"], user_input)
            if result["valid"] and result["value"] is not None:
                self.filled[current["slot"]] = result["value"]
            else:
                retries = self._retries.get(current["slot"], 0) + 1
                self._retries[current["slot"]] = retries
                if retries <= self.MAX_RETRIES_PER_SLOT:
                    return f"متوجه نشدم. لطفاً دوباره پاسخ بدید:\n{current['question']}"
                else:
                    # بعد از چند بار تلاش ناموفق، پاسخ خام را همانطور که هست ذخیره کن تا مکالمه گیر نکند
                    self.filled[current["slot"]] = user_input.strip()

        next_q = self._next_question()
        if next_q is None:
            self.state = "completed"
            return self.generate_json_response()
        return next_q["question"]

    def generate_json_response(self) -> str:
        response = {"intent": self.intent, "parameters": self.filled}
        return json.dumps(response, ensure_ascii=False, indent=2)


## اجرای تعاملی در کنسول


In [ ]:
chatbot = MultiTurnChatBot()

print("لطفاً ورودی اولیه خود را وارد کنید:")
user_input = input()
print(chatbot.process_input(user_input))

while chatbot.state != "completed":
    user_input = input()
    print(chatbot.process_input(user_input))


## تست خودکار (بدون input تعاملی)

In [ ]:
def simulate_conversation(turns: list[str]):
    bot = MultiTurnChatBot()
    for t in turns:
        print("کاربر:", t)
        reply = bot.process_input(t)
        print("ربات:", reply)
        print("-" * 40)
        if bot.state == "completed":
            break
    return bot

# مثال: انتقال وجه کارت به کارت که کاربر چند اطلاع را همزمان می‌دهد
_ = simulate_conversation([
    "می‌خوام ۲۰۰ هزار تومان کارت به کارت انتقال بدم به شماره کارت 6037991234567890",
    "علی",
    "رضایی",
    "امروز ساعت ۵ عصر",
    "بابت خرید",
    "123",
    "1234",
])
